# Multi-asset model stability

This notebook tests whether the strategy is stable under (1) XGBoost seed changes and (2) a switch between the Dukascopy and Oanda inputs. It deliberately uses one common calendar and fixed train/test dates. It does not modify `multi_asset_trading.ipynb`.

The default feature lookbacks are multiples of seven, so a trailing window contains whole weekly cycles when synthetic Saturday rows are retained.

In [18]:
import datetime
import json
from itertools import combinations
from pathlib import Path

import numpy as np
import polars as pl


In [19]:
# Experiment configuration. Keep these fixed while comparing sources.
CCYS = ['jpy', 'chf', 'cad', 'aud', 'nzd', 'eur', 'gbp']
MODEL_LOOKBACKS = [7, 14, 21, 28, 35, 42, 49, 98, 147, 196]
REMOVE_SYNTHETIC_SATURDAY = False

# Five seeds is a useful first pass. Increase only after the workflow is behaving as expected.
SEEDS = [15684, 15685, 15686, 15687, 15688]

COMMON_START = datetime.datetime(2016, 1, 3, tzinfo=datetime.timezone.utc)
TRAIN_END = datetime.datetime(2022, 12, 31, tzinfo=datetime.timezone.utc)
TEST_START = datetime.datetime(2023, 1, 1, tzinfo=datetime.timezone.utc)
TEST_END = datetime.datetime(2026, 8, 31, tzinfo=datetime.timezone.utc)

MODEL_PARAMS = {
    'n_estimators': 350,
    'learning_rate': 0.03,
    'max_depth': 4,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'early_stopping_rounds': 30,
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_jobs': -1,
}


In [20]:
# Reuse the current reader, feature, target, and XGBoost definitions without running
# the source notebook's data-loading or walk-forward cells.
source_notebook = Path('multi_asset_trading.ipynb')
if not source_notebook.exists():
    source_notebook = Path('src/notebooks/multi_asset_trading.ipynb')
if not source_notebook.exists():
    raise FileNotFoundError('Could not find multi_asset_trading.ipynb')

source_cells = json.loads(source_notebook.read_text(encoding='utf-8'))['cells']
shared = {'include_minors': False, 'dukas_data': True}
for index in (0, 3, 4, 5, 7):
    exec(''.join(source_cells[index]['source']), shared)

# Guard against an accidental currency-universe mismatch with the source notebook.
if shared['CCYS'] != CCYS:
    raise ValueError(f'Currency mismatch: {shared["CCYS"]} != {CCYS}')


In [21]:
def build_raw(source: str) -> pl.DataFrame:
    reader = shared['read_dukas_data'] if source == 'dukas' else shared['read_oanda_data']
    bars = {ccy: reader(ccy) for ccy in CCYS}
    renamed = [
        bars[ccy].select(
            'timestamp',
            pl.col('open').alias(f'{ccy}_open'),
            pl.col('high').alias(f'{ccy}_high'),
            pl.col('low').alias(f'{ccy}_low'),
            pl.col('close').alias(f'{ccy}_close'),
        )
        for ccy in CCYS
    ]
    return pl.concat(renamed, how='align_full').sort('timestamp').drop_nulls()

dukas_raw = build_raw('dukas')
oanda_raw = build_raw('oanda')
common_timestamps = dukas_raw.select('timestamp').join(oanda_raw.select('timestamp'), on='timestamp')

def prepare(source_raw: pl.DataFrame) -> pl.DataFrame:
    result = source_raw.join(common_timestamps, on='timestamp').filter(
        pl.col('timestamp').is_between(COMMON_START, TEST_END)
    )
    if REMOVE_SYNTHETIC_SATURDAY:
        result = result.filter(pl.col('timestamp').dt.weekday() != 6)
    return shared['add_model_features'](result, MODEL_LOOKBACKS)

frames = {'dukas': prepare(dukas_raw), 'oanda': prepare(oanda_raw)}


In [22]:
# Calendar and price sanity checks. These should be inspected before interpreting a result.
calendar_check = pl.concat([
    frame.group_by(pl.col('timestamp').dt.weekday().alias('weekday')).len()
    .with_columns(pl.lit(source).alias('source'))
    for source, frame in frames.items()
]).sort(['source', 'weekday'])
calendar_check

close_differences = frames['dukas'].select(['timestamp', *[f'{ccy}_close' for ccy in CCYS]]).join(
    frames['oanda'].select(['timestamp', *[pl.col(f'{ccy}_close').alias(f'{ccy}_oanda_close') for ccy in CCYS]]),
    on='timestamp',
)
close_difference_summary = pl.DataFrame({
    'ccy': CCYS,
    'mean_abs_difference_bp': [
        float((close_differences[f'{ccy}_oanda_close'] / close_differences[f'{ccy}_close'] - 1).abs().mean() * 1e4)
        for ccy in CCYS
    ],
})
close_difference_summary


ccy,mean_abs_difference_bp
str,f64
"""jpy""",1.156705
"""chf""",2.071465
"""cad""",1.21211
"""aud""",1.881675
"""nzd""",2.597202
"""eur""",0.787577
"""gbp""",1.423345


In [23]:
def realised_returns(weights: pl.DataFrame, data: pl.DataFrame) -> pl.DataFrame:
    labelled = data
    for ccy in CCYS:
        labelled = shared['calculate_target'](labelled, ccy)
    target_columns = [f'{ccy}_target' for ccy in CCYS]
    return (
        weights.join(labelled.select(['timestamp', *target_columns]), on='timestamp')
        .drop_nulls(target_columns)
        .with_columns(pl.sum_horizontal([
            pl.col(f'{ccy}_weight') * pl.col(f'{ccy}_target') for ccy in CCYS
        ]).alias('portfolio_return'))
        .sort('timestamp')
    )

def metrics(returns: pl.DataFrame) -> dict:
    values = returns['portfolio_return'].to_numpy()
    equity = np.prod(1 + values)
    volatility = np.std(values, ddof=1)
    return {
        'rows': len(values),
        'total_return': float(equity - 1),
        'annualised_return': float(equity ** (252 / len(values)) - 1),
        'annualised_volatility': float(volatility * np.sqrt(252)),
        'sharpe': None if volatility == 0 else float(values.mean() / volatility * np.sqrt(252)),
        'mean_return_bp': float(values.mean() * 1e4),
    }

def fit_models(source: str, seed: int):
    train = frames[source].filter(pl.col('timestamp').is_between(COMMON_START, TRAIN_END))
    params = {**MODEL_PARAMS, 'random_state': seed}
    models, validation_metrics, feature_columns = shared['train_xgboost_models'](
        train, model_params=params
    )
    return models, validation_metrics, feature_columns

def make_weights(models, feature_columns, source: str) -> pl.DataFrame:
    test = frames[source].filter(pl.col('timestamp').is_between(TEST_START, TEST_END))
    return shared['generate_weights'](models, test, feature_columns)


In [24]:
# For every trained model, score both its native input and the other source's input.
# This distinguishes seed instability, feature/source sensitivity, and price-only sensitivity.
records = []
weight_books = {}
validation_records = []

for train_source in ('dukas', 'oanda'):
    for seed in SEEDS:
        models, validation_metrics, feature_columns = fit_models(train_source, seed)
        validation_records.extend(
            validation_metrics.with_columns(
                pl.lit(train_source).alias('train_source'),
                pl.lit(seed).alias('seed'),
            ).to_dicts()
        )
        for execution_source in ('dukas', 'oanda'):
            weights = make_weights(models, feature_columns, execution_source)
            realised = realised_returns(weights, frames[execution_source])
            records.append({
                'train_source': train_source,
                'execution_source': execution_source,
                'seed': seed,
                **metrics(realised),
            })
            weight_books[(train_source, execution_source, seed)] = weights

results = pl.DataFrame(records).sort(['train_source', 'execution_source', 'seed'])
validation_results = pl.DataFrame(validation_records).sort(['train_source', 'seed', 'ccy'])
results


train_source,execution_source,seed,rows,total_return,annualised_return,annualised_volatility,sharpe,mean_return_bp
str,str,i64,i64,f64,f64,f64,f64,f64
"""dukas""","""dukas""",15684,1335,0.285985,0.048624,0.03186,1.506256,1.904367
"""dukas""","""dukas""",15685,1335,0.218297,0.037976,0.030149,1.251392,1.497176
"""dukas""","""dukas""",15686,1335,0.156285,0.02779,0.03225,0.866076,1.108384
"""dukas""","""dukas""",15687,1335,0.222749,0.03869,0.030334,1.266649,1.524717
"""dukas""","""dukas""",15688,1335,0.163888,0.029062,0.032238,0.904779,1.157478
…,…,…,…,…,…,…,…,…
"""oanda""","""oanda""",15684,1335,-0.025703,-0.004903,0.035541,-0.120549,-0.170017
"""oanda""","""oanda""",15685,1335,0.048625,0.009003,0.033255,0.28611,0.377562
"""oanda""","""oanda""",15686,1335,0.080599,0.01474,0.03071,0.491802,0.599342


In [25]:
# Stability within a source: independently seeded models should produce similar positions.
def weight_agreement(left: pl.DataFrame, right: pl.DataFrame) -> dict:
    joined = left.join(right, on='timestamp', suffix='_right')
    a = np.column_stack([joined[f'{ccy}_weight'].to_numpy() for ccy in CCYS])
    b = np.column_stack([joined[f'{ccy}_weight_right'].to_numpy() for ccy in CCYS])
    cosine = np.sum(a * b, axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    return {
        'rows': len(joined),
        'mean_weight_cosine': float(np.nanmean(cosine)),
        'weight_element_correlation': float(np.corrcoef(a.ravel(), b.ravel())[0, 1]),
        'same_weight_sign_fraction': float(np.mean(np.sign(a) == np.sign(b))),
    }

seed_agreement = []
for source in ('dukas', 'oanda'):
    for left_seed, right_seed in combinations(SEEDS, 2):
        seed_agreement.append({
            'source': source,
            'left_seed': left_seed,
            'right_seed': right_seed,
            **weight_agreement(
                weight_books[(source, source, left_seed)],
                weight_books[(source, source, right_seed)],
            ),
        })

seed_agreement = pl.DataFrame(seed_agreement)
seed_summary = seed_agreement.group_by('source').agg(
    pl.mean('mean_weight_cosine').alias('mean_pairwise_cosine'),
    pl.min('mean_weight_cosine').alias('min_pairwise_cosine'),
    pl.mean('weight_element_correlation').alias('mean_weight_correlation'),
    pl.mean('same_weight_sign_fraction').alias('mean_same_sign_fraction'),
).sort('source')

performance_summary = results.group_by(['train_source', 'execution_source']).agg(
    pl.len().alias('seed_runs'),
    pl.mean('sharpe').alias('mean_sharpe'),
    pl.std('sharpe').alias('sharpe_std'),
    pl.min('sharpe').alias('min_sharpe'),
    pl.max('sharpe').alias('max_sharpe'),
    pl.mean('mean_return_bp').alias('mean_daily_return_bp'),
).sort(['train_source', 'execution_source'])

performance_summary, seed_summary


(shape: (4, 8)
 ┌────────────┬────────────┬───────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
 │ train_sour ┆ execution_ ┆ seed_runs ┆ mean_sharp ┆ sharpe_st ┆ min_sharp ┆ max_sharp ┆ mean_dail │
 │ ce         ┆ source     ┆ ---       ┆ e          ┆ d         ┆ e         ┆ e         ┆ y_return_ │
 │ ---        ┆ ---        ┆ u32       ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ bp        │
 │ str        ┆ str        ┆           ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ ---       │
 │            ┆            ┆           ┆            ┆           ┆           ┆           ┆ f64       │
 ╞════════════╪════════════╪═══════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ dukas      ┆ dukas      ┆ 5         ┆ 1.15903    ┆ 0.269789  ┆ 0.866076  ┆ 1.506256  ┆ 1.438424  │
 │ dukas      ┆ oanda      ┆ 5         ┆ 0.524739   ┆ 0.209295  ┆ 0.295161  ┆ 0.694583  ┆ 0.648194  │
 │ oanda      ┆ dukas      ┆ 5         ┆ 0.092016   ┆ 0.237138  ┆ -

## How to read the results

- **Stable model:** narrow native-source Sharpe range across seeds and high pairwise weight cosine similarity.
- **Source robust model:** positive transfer rows (`dukas → oanda` and `oanda → dukas`) as well as positive native rows.
- **Unstable model:** wide seed Sharpe range or low same-source weight agreement. This is not a local-minimum issue; it means the fitted tree ensemble is sensitive to small perturbations.
- **Source sensitive model:** stable within a source but materially different across source-transfer rows. In that case, use feature-family ablations before tuning hyperparameters further.

Do not use this test period to select a winning parameter set. Once the design is fixed, reserve a later untouched period for the final assessment.